# Step 3: Pilot, English vs Hindi vs Kannada (where does the drop happen?)

**Inputs:** COCO dataset, `xling-spatial-code`, your **pilot CSV** uploaded as a Kaggle dataset (format in `pilot_template.csv`),
and optionally notebook 02's output (for English-tuned AdaptVis parameters).

For each model: accuracy and pair accuracy per language, then per-layer attention of the answer position over the image:
- `img_mass`: share of attention on image tokens at all,
- `box_frac`: of the image attention, the share on the subject + object boxes (dashed line = chance),
- `entropy_norm`: how spread out the image attention is.

Then the **flip analysis**: items correct in English but wrong in Kannada, with English and Kannada compared on the *same item*, layer by layer.

In [ ]:
# Shared setup for notebooks 02-04 (pinned transformers; do NOT run in the same session as notebook 01)
!pip install -q "transformers==4.57.1" "accelerate>=1.0" "huggingface_hub>=0.34"
import glob, os, sys
# The xling_spatial code: either a Kaggle dataset you uploaded (Add Input) or a git clone of your team repo.
REPO_URL = ""   # e.g. "https://github.com/<user>/<repo>.git" (leave empty if you attached the code as a Kaggle dataset)
def _find():
    return glob.glob("/kaggle/input/**/xling_spatial/__init__.py", recursive=True) + glob.glob("/kaggle/working/**/xling_spatial/__init__.py", recursive=True)
hits = _find()
if not hits and REPO_URL:
    !git clone -q {REPO_URL} /kaggle/working/repo
    hits = _find()
assert hits, "Set REPO_URL to your GitHub repo, or attach the 'xling-spatial-code' dataset (see README step 0)"
CODE_DIR = os.path.dirname(os.path.dirname(hits[0])); sys.path.insert(0, CODE_DIR); print("code:", CODE_DIR)

import torch, transformers, numpy as np, pandas as pd, matplotlib.pyplot as plt
from xling_spatial import data, runner, analysis, attention
from xling_spatial.models import VLM
from xling_spatial.tokenizer_stats import tokenizer_table
print(torch.__version__, transformers.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

COCO_ROOT = data.find_coco_root(); print("COCO:", COCO_ROOT)
WORK = "/kaggle/working"
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")   # needed only for gated models (Gemma 3)
except Exception:
    pass

def free(vlm):
    attention.uninstall()
    del vlm.model
    import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
PILOT_CSV = glob.glob("/kaggle/input/**/pilot*.csv", recursive=True)[0]   # or set the path yourself
pilot = data.load_pilot(PILOT_CSV)
LANGS = [l for l in ("en", "hi", "kn") if l in data.languages(pilot)]
print(len(pilot), "items", LANGS, pilot.variant.value_counts().to_dict())
boxes = data.load_coco_boxes(COCO_ROOT, pilot.image.unique().tolist(), f"{WORK}/boxes_pilot.json")
print("items with both boxes:", np.mean([r.subj in boxes[r.image] and r.obj in boxes[r.image] for r in pilot.itertuples()]))

In [ ]:
# Tokenizer check first (CPU only, a few seconds): report these numbers next to every accuracy.
tok_ids = {"Qwen2.5-VL": "Qwen/Qwen2.5-VL-3B-Instruct", "Gemma 3": "google/gemma-3-4b-it",
           "LLaVA-OV": "llava-hf/llava-onevision-qwen2-7b-ov-hf", "LLaVA-1.5": "llava-hf/llava-1.5-7b-hf"}
toks = {}
for k, mid in tok_ids.items():
    try:
        toks[k] = transformers.AutoTokenizer.from_pretrained(mid, token=HF_TOKEN)
    except Exception as e:
        print("skip", k, type(e).__name__)
tt = tokenizer_table(toks, pilot, LANGS); tt.to_csv(f"{WORK}/step3_tokenizers.csv", index=False); tt.round(3)

In [ ]:
MODELS = [
    dict(model_id="Qwen/Qwen2.5-VL-3B-Instruct"),
    # dict(model_id="Qwen/Qwen2.5-VL-7B-Instruct"),
    # dict(model_id="google/gemma-3-4b-it"),                     # gated: accept licence on HF + HF_TOKEN secret; runs fp32 over 2 T4s
    # dict(model_id="llava-hf/llava-onevision-qwen2-7b-ov-hf"),  # no box metrics (multi-crop image layout)
]
WEIGHTS = (0.5, 0.8, 1.0, 1.2, 1.5, 2.0)
PROMPT_MODE = "en_instruction"   # also run "native" once the Hindi/Kannada instructions are reviewed

all_res, all_att = [], []
for spec in MODELS:
    tag = spec["model_id"].split("/")[-1]
    vlm = VLM(**spec, hf_token=HF_TOKEN)
    out = f"{WORK}/step3_{tag}_{PROMPT_MODE}"
    r = runner.run(vlm, pilot, LANGS, COCO_ROOT, out, weights=WEIGHTS, boxes=boxes, save_maps=True, prompt_mode=PROMPT_MODE)
    a = pd.read_csv(f"{out}/attention.csv", dtype={"item_id": str})
    r["tag"], a["tag"] = tag, tag
    all_res.append(r); all_att.append(a)
    free(vlm)
res, att = pd.concat(all_res), pd.concat(all_att)

In [ ]:
s = analysis.summarize(res, by=("tag", "lang", "weight"))
base = s[np.isclose(s.weight, 1.0)].drop(columns="weight")
base.to_csv(f"{WORK}/step3_accuracy_base.csv", index=False)
base.round(3)   # yesno_mass << 1 means the model is not answering yes/no in that language: check before reading accuracy

In [ ]:
for tag, r in res.groupby("tag"):
    a = att[att.tag == tag]
    analysis.plot_layers(a, title=f"{tag} · all items · last-token attention"); plt.savefig(f"{WORK}/step3_{tag}_layers_all.png", dpi=150); plt.show()
    for lang in [l for l in LANGS if l != "en"]:
        flip, both = analysis.flip_sets(r, lang)
        print(f"{tag} {lang}: {len(flip)} flipped (en right, {lang} wrong), {len(both)} correct in both")
        if flip:
            analysis.plot_layers(a[a.lang.isin(["en", lang])], items=flip, title=f"{tag} · flipped items only (en vs {lang})")
            plt.savefig(f"{WORK}/step3_{tag}_layers_flip_{lang}.png", dpi=150); plt.show()
        for metric in ("box_frac", "img_mass", "entropy_norm"):
            if metric in a:
                d = analysis.paired_layer_diff(a, metric, lang, items=flip or None)
                d.to_csv(f"{WORK}/step3_{tag}_{lang}_{metric}_paired.csv", index=False)
        print("  ->", analysis.diagnose(r, a, lang))

In [ ]:
# Look at a few flipped items: where does the answer position look, English vs Kannada?
from PIL import Image
tag = res.tag.iloc[0]; r = res[res.tag == tag]
flip, _ = analysis.flip_sets(r, "kn") if "kn" in LANGS else (set(), None)
n_layers = att[att.tag == tag].layer.max() + 1
show = [n_layers // 4, n_layers // 2, 3 * n_layers // 4]
for item in sorted(flip)[:4]:
    row = pilot.set_index("item_id").loc[item]
    img = data.load_image(COCO_ROOT, row.image)
    for lang in ("en", "kn"):
        fp = f"{WORK}/step3_{tag}_{PROMPT_MODE}/maps/{item}__{lang}.npz"
        analysis.overlay(fp, img, layers=show, title=f"{item} [{lang}] {row['text_' + lang]}  (label={row.label}; cyan=subj, green=obj)")
        plt.show()

In [ ]:
# AdaptVis on Indian languages: English-tuned parameters vs the best fixed weight per language.
# With only ~50-150 pilot items, do NOT tune per language yet; do that on the 500-1000 item set with data.split_by_image().
params = glob.glob("/kaggle/input/**/adaptvis_params_*_en.json", recursive=True) + glob.glob(f"{WORK}/adaptvis_params_*_en.json")
p = pd.read_json(params[0], typ="series").to_dict() if params else {"weight1": 0.5, "weight2": 1.2, "threshold": 0.64}
print("using", p)
rows = []
for (tag, lang), r in res.groupby(["tag", "lang"]):
    av = runner.adaptvis_predictions(r, p["weight1"], p["weight2"], p["threshold"])
    rows.append({"tag": tag, "lang": lang, "base": r[np.isclose(r.weight, 1)].correct.mean(), "adaptvis_en_params": av.correct.mean(),
                 "share_weight1": (av.weight_used == p["weight1"]).mean(),
                 "best_fixed_weight_acc (oracle)": r.groupby("weight").correct.mean().max()})
av_table = pd.DataFrame(rows); av_table.to_csv(f"{WORK}/step3_adaptvis.csv", index=False); av_table.round(3)

In [ ]:
analysis.confidence_by_lang(res[res.tag == res.tag.iloc[0]]); plt.savefig(f"{WORK}/step3_confidence.png", dpi=150); plt.show()
analysis.accuracy_vs_weight(res[res.tag == res.tag.iloc[0]]); plt.savefig(f"{WORK}/step3_acc_vs_weight.png", dpi=150); plt.show()

### Reading the results (Review-1 decision table)
- **Flipped items show lower `box_frac` / higher entropy in Kannada at some layers** → "looks at the wrong part of the image".
  AdaptVis-style scaling (or a targeted version of it) is a fair candidate fix; check whether `best_fixed_weight_acc` closes the gap.
- **Attention looks the same, the answer still flips** → the relation is not encoded or not used. Go to activation patching (Stage 3, Phase II).
- **`img_mass` itself drops in Kannada** → the model is busy with the (longer) text; compare with the tokenizer table before blaming morphology.
- Lower confidence in Kannada means AdaptVis will mostly pick `weight1`: report `share_weight1`.